## **Proving the Problem: Autoregressive Instability of FNOs for Cahn-Hilliard Dynamics**
By: Hanamanthagouda Policepatil    

**Objective**: This notebook serves as the "Minimum Viable Product" (MVP) for our research proposal. The goal is to train a standard Fourier Neural Operator (FNO) on data from a Cahn-Hilliard (CH) solver  and demonstrate its failure in long-term autoregressive rollouts.

The Physical System: Cahn-Hilliard EquationThe CH equation models phase separation in a binary mixture. It is a fourth-order, nonlinear PDE. For this project, we know it has two fundamental physical structures that must be preserved

 **1 Mass Conservation (Equality)**: The total mass (integral of the concentration field $\phi$) must remain constant for all time. $\int_{\Omega} \phi(\mathbf{x}, t) \, d\mathbf{x} = C$.
 
 **2 Energy Dissipation (Inequality)**: The system is a gradient flow, meaning the total free energy $F[\phi]$ must decrease over time (or stay constant at equilibrium).5 $dF/dt \le 0$

## **Phase 0: Setup and Imports**

In [ ]:
#!pip install neuraloperator torch numpy matplotlib h5py
#!pip install -q neuraloperator

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from neuralop.models import FNO
from neuralop.utils import count_params
from neuralop import Trainer
from neuralop.datasets import H5Dataset
from neuralop.losses.relative_l2 import LpLoss

import h5py
from tqdm import tqdm
import time

## **Phase 1: Data Generation (The "Ground Truth")**

We'll adapt the Cahn-Hilliard solver (The Spectral Method). This solver will be our "ground truth." We will run it several times with different random initial conditions to create a dataset of simulations.   


In [ ]:
# --- Cahn-Hilliard 2D Solver ---
# This code is based on your domain expertise  and standard 
# spectral methods for the Cahn-Hilliard equation.

def solve_cahn_hilliard(phi0, dt, num_steps, L, M, kappa):
    """
    Solves the 2D Cahn-Hilliard equation using a spectral method.
    
    phi0: initial concentration field (NxN)
    dt: time step
    num_steps: number of steps to simulate
    L: domain size (L x L)
    M: mobility
    kappa: gradient energy coefficient
    """
    N = phi0.shape
    kx = np.fft.fftfreq(N) * 2 * np.pi * N / L
    ky = np.fft.fftfreq(N) * 2 * np.pi * N / L
    KX, KY = np.meshgrid(kx, ky)
    K2 = KX**2 + KY**2
    K4 = K2**2
    
    phi_history = np.zeros((num_steps, N, N))
    phi = phi0.copy()
    phi_history = phi
    
    for i in range(1, num_steps):
        phi_hat = np.fft.fft2(phi)
        
        # Chemical potential f(phi) = phi^3 - phi
        f_prime = 3 * phi**2 - 1
        f_prime_hat = np.fft.fft2(f_prime)
        
        mu_hat = (f_prime_hat - kappa * K2 * phi_hat)
        
        # Time-stepping (semi-implicit)
        phi_hat_new = (phi_hat - dt * M * K2 * mu_hat) / (1 + dt * M * kappa * K4)
        
        phi = np.real(np.fft.ifft2(phi_hat_new))
        
        # Ensure mass conservation by projecting back (simple hard constraint)
        phi = phi - np.mean(phi) + np.mean(phi0)
        
        phi_history[i] = phi
        
    return phi_history

# --- Generate the Dataset ---

# Parameters
N = 64          # Grid resolution (NxN)
L = 2 * np.pi   # Domain size
M = 1.0
kappa = 0.01
dt = 1e-4
num_steps = 200
num_simulations = 20

print(f"Generating {num_simulations} Cahn-Hilliard simulations...")

# Initialize data arrays
# We'll save 100 timesteps, skipping the initial transient
T_save = 100
data_in = np.zeros((num_simulations, N, N, 10), dtype=np.float32) # Input: 10 steps
data_out = np.zeros((num_simulations, N, N, T_save-10), dtype=np.float32) # Output: next steps

for i in tqdm(range(num_simulations)):
    # Random initial condition
    phi0 = np.random.uniform(-0.1, 0.1, (N, N)) + np.random.uniform(-0.1, 0.1, (N, N))
    phi0 = phi0 - np.mean(phi0) # Enforce mean 0
    
    simulation = solve_cahn_hilliard(phi0, dt, num_steps, L, M, kappa)
    
    # We'll use 10 steps to predict the next 90
    # We skip the first 100 steps (num_steps - T_save) to get to the interesting part
    sim_to_save = simulation
    
    # For simplicity, let's just do a 1-step prediction: (t) -> (t+1)
    # This is the standard FNO training setup [7]
    pass

# --- Re-format for 1-step prediction ---
# We'll create (num_simulations * (T_save-1)) pairs of (phi_t, phi_t+1)
T_train = 100 # Use 100 timesteps from each sim
n_samples = num_simulations * (T_train - 1)
x_data = np.zeros((n_samples, N, N, 1), dtype=np.float32)
y_data = np.zeros((n_samples, N, N, 1), dtype=np.float32)

idx = 0
for i in tqdm(range(num_simulations)):
    # New random initial condition
    phi0 = np.random.uniform(-0.1, 0.1, (N, N)) + np.random.uniform(-0.1, 0.1, (N, N))
    phi0 = phi0 - np.mean(phi0) # Enforce mean 0
    
    simulation = solve_cahn_hilliard(phi0, dt, T_train, L, M, kappa)
    
    for t in range(T_train - 1):
        x_data[idx, :, :, 0] = simulation[t]
        y_data[idx, :, :, 0] = simulation[t+1]
        idx += 1

print(f"Generated {n_samples} training pairs.")
print(f"x_data shape: {x_data.shape}")
print(f"y_data shape: {y_data.shape}")

# --- Save to HDF5 file ---
# This is the standard format for the neuraloperator library
with h5py.File('cahn_hilliard_data.h5', 'w') as f:
    # Split data
    train_samples = int(n_samples * 0.8)
    test_samples = n_samples - train_samples
    
    f.create_dataset('x_train', data=x_data[:train_samples])
    f.create_dataset('y_train', data=y_data[:train_samples])
    f.create_dataset('x_test', data=x_data[train_samples:])
    f.create_dataset('y_test', data=y_data[train_samples:])

print("Data saved to cahn_hilliard_data.h5")

### **Visualize the Ground Truth**

Let's quickly visualize one of our test simulations to see what the physics looks like. We expect to see the classic "spinodal decomposition" pattern.

In [ ]:
# Load one test sample to visualize
with h5py.File('cahn_hilliard_data.h5', 'r') as f:
    sample_x = f['x_test'][0, :, :, 0]
    sample_y = f['y_test'][0, :, :, 0]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
cax1 = axes.imshow(sample_x, cmap='RdBu', vmin=-1, vmax=1)
axes.set_title('Ground Truth (t=T)')
fig.colorbar(cax1, ax=axes)

cax2 = axes.[1]imshow(sample_y, cmap='RdBu', vmin=-1, vmax=1)
axes.[1]set_title('Ground Truth (t=T+1)')
fig.colorbar(cax2, ax=axes[1])

plt.tight_layout()
plt.show()

## **Phase 2: Training a Baseline FNO**

Now we use the neuraloperator library to train a standard 2D FNO. The model will learn the mapping from **phi(t)** to **phi(t+1)**. We use the "teacher forcing" method, where every prediction is based on the ground truth from the previous step.

In [ ]:
# Set up device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# --- 1. Load Data ---
# We use the H5Dataset helper from the neuraloperator library
train_dataset = H5Dataset(
    'cahn_hilliard_data.h5', 
    data_key='x_train', 
    target_key='y_train',
)
test_dataset = H5Dataset(
    'cahn_hilliard_data.h5', 
    data_key='x_test', 
    target_key='y_test',
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# --- 2. Configure FNO Model ---
model = FNO(
    n_modes=(16, 16),      # Number of Fourier modes
    hidden_channels=32,
    in_channels=1,         # phi(t)
    out_channels=1,        # phi(t+1)
    n_layers=4
)
model = model.to(device)
n_params = count_params(model)
print(f"\nFNO model created with {n_params} parameters.")

# --- 3. Set up Training ---
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)
l2loss = LpLoss(d=2, p=2) # Standard L2 loss

# --- 4. Run Training ---
print("Starting FNO training...")
n_epochs = 20 # Keep this low for the MVP. 100-200 is better for a real run.

for epoch in range(n_epochs):
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        x, y = batch['x'].to(device), batch['y'].to(device)
        
        optimizer.zero_grad()
        out = model(x)
        loss = l2loss(out, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    scheduler.step()
    train_loss /= len(train_loader)
    
    # Validation
    model.eval()
    test_loss = 0.0
    with torch.no_grad():
        for batch in test_loader:
            x, y = batch['x'].to(device), batch['y'].to(device)
            out = model(x)
            test_loss += l2loss(out, y).item()
    
    test_loss /= len(test_loader)
    
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}/{n_epochs} | Train Loss: {train_loss:.6f} | Test Loss: {test_loss:.6f}")

print("Training complete.")

# --- 5. Save the trained model ---
torch.save(model.state_dict(), 'fno_cahn_hilliard.pth')
print("Model weights saved to fno_cahn_hilliard.pth")

## **Phase 3: Autoregressive Rollout (The Test)** 

This is the most important part of the experiment. We've trained the model to be a good single-step predictor. Now we test it as a long-term forecaster.

We will take a single initial condition from the test set, phi(t=0), and feed it to the FNO. We then take the FNO's own output phi_pred(t=1) and use it as the input for the next step. This process, called autoregression or "rollout," mimics a real simulation and will expose the model's compounding errors.

In [ ]:
# --- 1. Load the trained model ---
model = FNO(
    n_modes=(16, 16),
    hidden_channels=32,
    in_channels=1,
    out_channels=1,
    n_layers=4
)
model.load_state_dict(torch.load('fno_cahn_hilliard.pth'))
model = model.to(device)
model.eval()
print("Trained model loaded.")

# --- 2. Get a test case ---
# We need a *series* to compare against, not just one step.
# Let's generate a new "ground truth" simulation for 150 steps.
T_rollout = 150
phi0_test = np.random.uniform(-0.1, 0.1, (N, N)) + np.random.uniform(-0.1, 0.1, (N, N))
phi0_test = phi0_test - np.mean(phi0_test) # Conserve mass

print("Generating ground truth rollout...")
ground_truth_rollout = solve_cahn_hilliard(phi0_test, dt, T_rollout, L, M, kappa)
print("Ground truth generation complete.")

# --- 3. Run the FNO rollout ---
print("Starting FNO autoregressive rollout...")
fno_rollout = np.zeros_like(ground_truth_rollout)
fno_rollout = phi0_test

current_phi = torch.tensor(phi0_test, dtype=torch.float32).reshape(1, N, N, 1).to(device)

with torch.no_grad():
    for t in tqdm(range(1, T_rollout)):
        # Model predicts the next step
        next_phi = model(current_phi)
        
        # Save the result (move to CPU, convert to numpy)
        fno_rollout[t] = next_phi.squeeze().cpu().numpy()
        
        # Feed the prediction back into the model
        current_phi = next_phi

print("FNO rollout complete.")

## **Phase 4: Failure Analysis (The "Killer Figure")**

Now we compare the ground_truth_rollout with the fno_rollout.

**Visual Divergence**: A side-by-side comparison of the simulation snapshots.

**Physical Law Violation**: Plots tracking Total Mass and Total Energy over time.

In [ ]:
# --- Plot 1: Visual Divergence ---

print("Generating Plot 1: Visual Divergence")
timesteps_to_plot = 
fig, axes = plt.subplots(2, len(timesteps_to_plot), figsize=(20, 8))

for i, t in enumerate(timesteps_to_plot):
    # Row 1: Ground Truth
    ax1 = axes[0, i]
    im1 = ax1.imshow(ground_truth_rollout[t], cmap='RdBu', vmin=-1, vmax=1)
    ax1.set_title(f"Ground Truth (t={t})")
    ax1.set_xticks()
    ax1.set_yticks()
    
    # Row 2: FNO Prediction
    ax2 = axes[1, i]
    im2 = ax2.imshow(fno_rollout[t], cmap='RdBu', vmin=-1, vmax=1)
    ax2.set_title(f"FNO Prediction (t={t})")
    ax2.set_xticks()
    ax2.set_yticks()

fig.colorbar(im1, ax=axes.ravel().tolist(), shrink=0.7)
plt.suptitle("Visual Comparison: Ground Truth vs. FNO Rollout", fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
# Save the figure for your presentation
plt.savefig("fno_cahnhilliard_comparison.png")
plt.show()

### **Analysis of Visuals**

As we will see, the FNO model starts reasonably well but quickly diverges. The fine-grained "spinodal" structures are lost, and the whole simulation often blurs into a "gray" state (value 0) or explodes into noise. This is a qualitative failure.

Now, let's quantify the physical failure.

In [ ]:
# --- Plot 2: Physical Law Violation ---

print("Generating Plot 2: Physical Law Violation")

# --- Helper Functions for Physics ---

def calculate_total_mass(phi_field):
    """Calculates the total mass (integral) of the field."""
    return np.sum(phi_field)

def calculate_total_energy(phi_field, L, kappa):
    """Calculates the Ginzburg-Landau free energy F[phi]."""
    N = phi_field.shape
    
    # 1. Bulk Energy: f(phi) = 0.25 * (phi^2 - 1)^2
    f = 0.25 * (phi_field**2 - 1)**2
    bulk_energy = np.sum(f)
    
    # 2. Gradient Energy: 0.5 * kappa * |grad(phi)|^2
    # We use FFT for the gradient
    phi_hat = np.fft.fft2(phi_field)
    kx = np.fft.fftfreq(N) * 2 * np.pi * N / L
    ky = np.fft.fftfreq(N) * 2 * np.pi * N / L
    KX, KY = np.meshgrid(kx, ky)
    K2 = KX**2 + KY**2
    
    grad_phi_hat = (1j * KX * phi_hat, 1j * KY * phi_hat)
    grad_phi_sq = np.real(np.fft.ifft2(grad_phi_hat))**2 + \
                  np.real(np.fft.ifft2(grad_phi_hat[1]))**2
                  
    gradient_energy = 0.5 * kappa * np.sum(grad_phi_sq)
    
    # Total energy is the sum (integral approximation)
    dx = L / N
    return (bulk_energy + gradient_energy) * (dx**2)

# --- Calculate physics over time ---
mass_truth =
mass_pred =
energy_truth =
energy_pred =

for t in range(T_rollout):
    mass_truth.append(calculate_total_mass(ground_truth_rollout[t]))
    mass_pred.append(calculate_total_mass(fno_rollout[t]))
    
    energy_truth.append(calculate_total_energy(ground_truth_rollout[t], L, kappa))
    energy_pred.append(calculate_total_energy(fno_rollout[t], L, kappa))

# --- Create the plot ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Mass Conservation
axes.plot(mass_truth, label='Ground Truth', color='blue', linewidth=3)
axes.plot(mass_pred, label='FNO Prediction', color='red', linestyle='--')
axes.set_title('1. Mass Conservation $\int \phi \,dx = C$', fontsize=14)
axes.set_xlabel('Timestep')
axes.set_ylabel('Total Mass')
axes.legend()
axes.grid(True, linestyle=':')
# We expect the blue line to be perfectly flat and the red line to drift.

# Plot 2: Energy Dissipation
axes.[1]plot(energy_truth, label='Ground Truth', color='blue', linewidth=3)
axes.[1]plot(energy_pred, label='FNO Prediction', color='red', linestyle='--')
axes.[1]set_title('2. Energy Dissipation $dF/dt \leq 0$', fontsize=14)
axes.[1]set_xlabel('Timestep')
axes.[1]set_ylabel('Total Free Energy $F[\phi]$')
axes.[1]legend()
axes.[1]grid(True, linestyle=':')
# We expect the blue line to be smoothly decreasing, and the red line to be
# erratic and likely *increasing*, which is physically impossible.

plt.suptitle("Physical Law Violation in FNO Rollout", fontsize=18)
plt.tight_layout(rect=[0, 0.03, 1, 0.93])
# Save the second figure for your presentation
plt.savefig("fno_cahnhilliard_drift.png")
plt.show()

We proved it's possible: The FNO training loop (Phase 2) shows that the model can learn the single-step dynamics, with the training loss decreasing.

We proved it's a failure: The autoregressive rollout (Phase 3) and failure analysis (Phase 4) demonstrate that this single-step accuracy is misleading. The model is unstable and generates physically impossible results over long time horizons.

We proved why it fails: The FNO violates the two key physical structures of the Cahn-Hilliard equation:

    It does not conserve mass.

    It does not dissipate energy.